<a href="https://colab.research.google.com/github/anshuldeepbajpai-dhoni/movie-recommendation-system/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-07-19 14:26:39--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  67.0MB/s    in 0.4s    

2026-07-19 14:26:40 (67.0 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# add your code here - consider creating a newimport pandas as pd
from sklearn.neighbors import NearestNeighbors

# Load datasets
books = pd.read_csv(
    "BX-Books.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    low_memory=False
)

ratings = pd.read_csv(
    "BX-Book-Ratings.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    low_memory=False
)

# Keep required columns
books = books[['ISBN', 'Book-Title']]
ratings = ratings[['User-ID', 'ISBN', 'Book-Rating']]

# --------------------------------------------------
# Remove users with fewer than 200 ratings
# --------------------------------------------------

user_counts = ratings['User-ID'].value_counts()
ratings = ratings[
    ratings['User-ID'].isin(user_counts[user_counts >= 200].index)
]

# --------------------------------------------------
# Remove books with fewer than 100 ratings
# --------------------------------------------------

book_counts = ratings['ISBN'].value_counts()
ratings = ratings[
    ratings['ISBN'].isin(book_counts[book_counts >= 100].index)
]

# --------------------------------------------------
# Merge datasets
# --------------------------------------------------

df = ratings.merge(books, on="ISBN")

# --------------------------------------------------
# Pivot Table
# --------------------------------------------------

book_pivot = df.pivot_table(
    index="Book-Title",
    columns="User-ID",
    values="Book-Rating"
).fillna(0)

# --------------------------------------------------
# KNN Model
# --------------------------------------------------

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

model.fit(book_pivot.values)

# --------------------------------------------------
# Recommendation Function
# --------------------------------------------------

def get_recommends(book=""):

    if book not in book_pivot.index:
        return None

    distances, indices = model.kneighbors(
        [book_pivot.loc[book].values],
        n_neighbors=6
    )

    recommendations = []

    for distance, idx in zip(
        distances.flatten()[1:],
        indices.flatten()[1:]
    ):

        recommendations.append([
            book_pivot.index[idx],
            float(distance)
        ])

    recommendations.reverse()

    return [book, recommendations]

In [7]:
def get_recommends(book=""):

    if book not in book_pivot.index:
        return None

    distances, indices = model.kneighbors(
        [book_pivot.loc[book].values],
        n_neighbors=6
    )

    recommended_books = []

    for distance, idx in zip(
        distances.flatten()[1:],
        indices.flatten()[1:]
    ):
        recommended_books.append([
            book_pivot.index[idx],
            float(distance)
        ])

    recommended_books.reverse()

    return [book, recommended_books]

In [8]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['Bel Canto: A Novel', 0.8247874880681115], ['The Notebook', 0.8236682900571164], ['The Joy Luck Club', 0.8198604785739199], ["The Pilot's Wife : A Novel", 0.8192678336213406], ['The Lovely Bones: A Novel', 0.7234864549790632]]]
You haven't passed yet. Keep trying!
